In [1]:
import warnings
warnings.filterwarnings("ignore")
!pip install yahooquery
!pip install xlrd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.5/216.5 kB 12.2 MB/s eta 0:00:00
  Attempting uninstall: cffi
    Found existing installation: cffi 1.16.0
    Uninstalling cffi-1.16.0:
      Successfully uninstalled cffi-1.16.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.6/96.6 kB 5.8 MB/s eta 0:00:00


In [2]:
# Cell 2: Updated imports and get_stock_data function
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from yahooquery import Ticker
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
bot_token = user_secrets.get_secret("TELEGRAM_TOKEN")


def get_stock_data(stock):
    t = Ticker(stock)
    
    # Extract data from different modules
    summary = t.summary_detail.get(stock, {})
    key_stats = t.key_stats.get(stock, {})
    fin_data = t.financial_data.get(stock, {})
    asset_profile = t.asset_profile.get(stock, {})
    
    # Retrieve stock data with fallback to NaN
    stock_price = summary.get('regularMarketPreviousClose', np.nan)
    
    # Check for 'trailingPE', then 'forwardPE', and fallback to NaN
    pe_ratio = summary.get('trailingPE') if summary.get('trailingPE') is not None else summary.get('forwardPE', np.nan)
    
    pb_ratio = key_stats.get('priceToBook', np.nan)
    
    # Handle dividend yield (convert to percentage if available)
    dividend_yield = summary.get('dividendYield', 0)
    if dividend_yield is not None:
        dividend_yield *= 100
    
    # Default industry to 'Unknown' if missing
    industry = asset_profile.get('industry', 'Unknown')
    
    # Get the mean and median recommendations
    mean_recommendation = fin_data.get('targetMeanPrice', np.nan)
    median_recommendation = fin_data.get('targetMedianPrice', np.nan)
    
    # Calculate financial intrinsic value
    growth_rate = fin_data.get('earningsGrowth', np.nan)
    eps = key_stats.get('trailingEps', np.nan)
    
    if growth_rate is not None and eps is not None:
        financial_intrinsic_value = eps * (1 + growth_rate) * pe_ratio
    else:
        financial_intrinsic_value = np.nan
    
    return stock_price, pe_ratio, pb_ratio, dividend_yield, mean_recommendation, median_recommendation, financial_intrinsic_value, industry
    

In [ ]:
# Cell 3: Telegram function
import requests

def send_telegram_message(message):
    chat_id = user_secrets.get_secret("chat_id")
    url = f"https://api.telegram.org/bot{bot_token}/sendMessage"

    params = {"chat_id": chat_id, "text": message}

    response = requests.post(url, params=params)
    if response.status_code == 200:
        print("Message sent successfully!")
    else:
        print(f"Failed to send message. Status code: {response.status_code}")

In [4]:
# Cell 4: Get PE and PB averages
from bs4 import BeautifulSoup
import requests
import numpy as np
import warnings
warnings.filterwarnings('ignore')

url = 'http://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/pedata.html'

def get_pe_averages():
    # Disable SSL verification
    response = requests.get(url, verify=False)
    soup = BeautifulSoup(response.content, 'html.parser')
    table = soup.find_all('table')[0]  # Find the first table
    rows = table.find_all('tr')
    pe_averages = {}
    for row in rows[1:]:  # Skip the header row
        items = row.find_all('td')
        industry = items[0].text.strip()
        pe = items[3].text.strip()
        pe_averages[industry] = pe
    return pe_averages

def get_pe_average(industry, pe_averages):
    return pe_averages.get(industry, np.nan)

# Get the PE averages
pe_averages = get_pe_averages()

#get the pb ratios of the industries
url = 'https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/pbvdata.html'
def get_pb_averages():
    # Disable SSL verification
    response = requests.get(url, verify=False)
    soup = BeautifulSoup(response.content, 'html.parser')
    table = soup.find_all('table')[0]  # Find the first table
    rows = table.find_all('tr')
    pb_averages = {}
    for row in rows[1:]:  # Skip the header row
        items = row.find_all('td')
        industry = items[0].text.strip()
        pb = items[2].text.strip()
        pb_averages[industry] = pb
    return pb_averages

def get_pb_average(industry, pb_averages):
    return pb_averages.get(industry, np.nan)


#try to get the pb ratio of the industry
pb_averages = get_pb_averages()

# clean the pe_averages, some of the keys have \n\t\t in them

#find all instances of \n\t\t and replace with empty string
pe_averages = {industry.replace('\n\t\t', ''): pe for industry, pe in pe_averages.items()}
# find all instances of more than one space and replace with one space
pe_averages = {industry.replace('        ', ' '): pe for industry, pe in pe_averages.items()}
pe_averages = {industry.strip(): pe for industry, pe in pe_averages.items()}

#clean the pb_averages, some of the keys have \n\t\t in them

#find all instances of \n\t\t and replace with empty string
pb_averages = {industry.replace('\n\t\t', ''): pb for industry, pb in pb_averages.items()}
# find all instances of more than one space and replace with one space
pb_averages = {industry.replace('      ', ' '): pb for industry, pb in pb_averages.items()}
pb_averages = {industry.strip(): pb for industry, pb in pb_averages.items()}

print(f"Loaded {len(pe_averages)} PE averages and {len(pb_averages)} PB averages")

Loaded 97 PE averages and 97 PB averages


In [5]:
# Cell 5: Main stock data collection loop (without sending to Telegram to avoid spam)

stocks = ['AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META', 'HSY', 'KO', 'PEP', 'NKE', 'V', 'INTC', 'NVDA', 'AMD', 'IBM', 'ORCL','ASML']

# Get information for each stock
results = []
for stock in stocks:
    stock_price, pe_ratio, pb_ratio, dividend_yield, mean_rec, median_rec, financial_iv, industry = get_stock_data(stock)
    #map the industry to the correct key
    #map Semiconductors to Semiconductor
    if industry == 'Semiconductors' or industry == 'Semiconductor Equipment & Materials':
        industry = 'Semiconductor'
    #map Software-Infrastructure to Software (System & Application)
    if industry == 'Software - Infrastructure' or industry == 'Information Technology Services':
        industry = 'Software (System & Application)'
    #map Consumer Electronics to Software (Entertainment)
    if industry == 'Consumer Electronics':
        industry = 'Software (Entertainment)'
    #map Internet Content & Information to Software (Internet)
    if industry == 'Internet Content & Information' or industry == 'Internet Retail':
        industry = 'Software (Internet)'
    #map Confectioners to Food Processing
    if industry == 'Confectioners':
        industry = 'Food Processing'
    #map Beverages - Non-Alcoholic to Beverages (Soft)
    if industry == 'Beverages - Non-Alcoholic':
        industry = 'Beverage (Soft)'
    #map Credit Services to Financial Svcs. (Non-bank & Insurance)	
    if industry == 'Credit Services':
        industry = 'Financial Svcs. (Non-bank & Insurance)'
    #map Footwear & Accessories to Apparel
    if industry == 'Footwear & Accessories':
        industry = 'Apparel'
    pe_average = get_pe_average(industry, pe_averages)
    pb_average = get_pb_average(industry, pb_averages)
    results.append([stock, industry, stock_price, pe_ratio, pb_ratio, dividend_yield, pe_average, pb_average, mean_rec, median_rec, financial_iv])

print(f"Collected data for {len(results)} stocks")
df = pd.DataFrame(results, columns=['Stock', 'Industry', 'Price', 'PE Ratio', 'PB Ratio', 'Dividend Yield', 'Industry PE', 'Industry PB', 'Target Mean Price', 'Target Median Price', 'Financial Intrinsic Value'])

Collected data for 16 stocks


In [6]:
# Cell 7: Process recommendations (printing only, not sending to Telegram)

for index, row in df.iterrows():
    stock = row['Stock']
    industry = row['Industry']
    price = row['Price']
    pe_ratio = row['PE Ratio']
    pb_ratio = row['PB Ratio']
    div_yield = row['Dividend Yield']
    pe_avg = float(row['Industry PE']) if not pd.isna(row['Industry PE']) else float('nan')
    pb_avg = float(row['Industry PB']) if not pd.isna(row['Industry PB']) else float('nan')
    target_mean_price = row['Target Mean Price']
    target_median_price = row['Target Median Price']
    intrinsic_val = row['Financial Intrinsic Value']
    
    # Format the message
    message = f"Stock: {stock}\n"
    message += f"Industry: {industry}\n"
    message += f"Price: ${price:.2f}\n"
    message += f"PE Ratio: {pe_ratio:.2f} (Industry Avg: {pe_avg:.2f})\n"
    message += f"PB Ratio: {pb_ratio:.2f} (Industry Avg: {pb_avg:.2f})\n"
    message += f"Dividend Yield: {div_yield:.2f}%\n"
    message += f"Target Mean Price: ${target_mean_price:.2f}\n"
    message += f"Target Median Price: ${target_median_price:.2f}\n"
    message += f"Financial Intrinsic Value: ${intrinsic_val:.2f}\n"
    
    # Recommendation logic
    if pe_ratio < pe_avg and pb_ratio < pb_avg:
        message += "Recommendation: Buy\n"
    elif pe_ratio > pe_avg and pb_ratio > pb_avg:
        message += "Recommendation: Sell\n"
    else:
        message += "Recommendation: Hold\n"
    print(message)
    send_telegram_message(message)


Stock: AAPL
Industry: Software (Entertainment)
Price: $290.55
PE Ratio: 35.34 (Industry Avg: 70.74)
PB Ratio: 40.16 (Industry Avg: 9.10)
Dividend Yield: 0.37%
Target Mean Price: $312.48
Target Median Price: $310.00
Financial Intrinsic Value: $355.14
Recommendation: Hold

Message sent successfully!
Stock: MSFT
Industry: Software (System & Application)
Price: $403.41
PE Ratio: 23.64 (Industry Avg: 122.49)
PB Ratio: 7.12 (Industry Avg: 9.14)
Dividend Yield: 0.92%
Target Mean Price: $560.95
Target Median Price: $555.00
Financial Intrinsic Value: $490.34
Recommendation: Buy

Message sent successfully!
Stock: AMZN
Industry: Software (Internet)
Price: $244.19
PE Ratio: 31.65 (Industry Avg: 162.98)
PB Ratio: 5.79 (Industry Avg: 10.86)
Dividend Yield: 0.00%
Target Mean Price: $312.71
Target Median Price: $317.00
Financial Intrinsic Value: $416.02
Recommendation: Buy

Message sent successfully!
Stock: GOOGL
Industry: Software (Internet)
Price: $364.26
PE Ratio: 27.18 (Industry Avg: 162.98)
PB Ra

In [7]:
import pandas as pd
import numpy as np
from yahooquery import Ticker
import requests

In [8]:
def fetch_mcclellan_oscillator(days=60):
    url = "https://www.mcoscillator.com/data/osc_data/OSC-DATA.xls"
    headers = {"User-Agent": "Mozilla/5.0"}
    
    try:
        response = requests.get(url, headers=headers, timeout=30)
        # Load the XLS - the data usually starts around row 6
        df_mcc = pd.read_excel(response.content, engine='xlrd', skiprows=6)
        
        # Column 0 is Date, Column 9 is McClellan Oscillator
        df_mcc = df_mcc.iloc[:, [0, 9]]
        df_mcc.columns = ['Date', 'McClellan_Oscillator']
        df_mcc['Date'] = pd.to_datetime(df_mcc['Date'])
        df_mcc = df_mcc.dropna().set_index('Date')
        
        # Return the last 'n' days to build the chart trend
        return df_mcc['McClellan_Oscillator'].tail(days)
    except Exception as e:
        print(f"⚠ McClellan fetch error: {e}")
        # Fallback to synthetic if the site is down
        dates = pd.date_range(end=pd.Timestamp.today(), periods=days, freq='B')
        return pd.Series(np.random.normal(0, 15, len(dates)), index=dates, name='McClellan_Oscillator')


In [9]:

import matplotlib.pyplot as plt

def send_telegram_with_chart(message, series):
    TELEGRAM_CHAT_ID = "972451274"
    if not bot_token or not TELEGRAM_CHAT_ID:
        print("⚠ Telegram credentials missing.")
        return

    # Create the plot
    plt.figure(figsize=(10, 5))
    plt.style.use('dark_background')
    
    # Plot the line
    series.plot(color='#00d1ff', linewidth=2, label='McClellan Oscillator')
    
    # Add overbought/oversold levels
    plt.axhline(100, color='red', linestyle='--', alpha=0.5)
    plt.axhline(-100, color='green', linestyle='--', alpha=0.5)
    plt.axhline(0, color='white', linewidth=0.8)
    
    plt.title(f"McClellan Oscillator Trend (Last {len(series)} Days)")
    plt.grid(alpha=0.1)
    
    # Save to buffer
    chart_path = "temp_chart.png"
    plt.savefig(chart_path, bbox_inches='tight')
    plt.close()

    # Send to Telegram
    url = f"https://api.telegram.org/bot{bot_token}/sendPhoto"
    with open(chart_path, 'rb') as photo:
        payload = {
            'chat_id': TELEGRAM_CHAT_ID,
            'caption': message,
            'parse_mode': 'HTML'
        }
        files = {'photo': photo}
        try:
            r = requests.post(url, data=payload, files=files)
            if r.status_code == 200:
                print("✓ Chart and Message sent!")
            else:
                print(f"✗ Failed: {r.text}")
        except Exception as e:
            print(f"✗ Connection Error: {e}")

In [10]:
# 1. Fetch data (ensure at least 14 days for context)
mcclellan_recent = fetch_mcclellan_oscillator(days=14)

# 2. Extract values for the alert logic
current_val = mcclellan_recent.iloc[-1]
prior_val = mcclellan_recent.iloc[-2]
recent_date = mcclellan_recent.index[-1].strftime('%Y-%m-%d')

# 3. Detect the "Oversold to Overbought" Thrust (-70 to +70)
thrust_detected = prior_val < -70 and current_val > 70
thrust_alert = ""

if thrust_detected:
    thrust_alert = "\n\n🚀 <b>BREADTH THRUST ALERT</b>\n"
    thrust_alert += f"Extreme reversal: {prior_val:+.1f} → {current_val:+.1f}\n"
    thrust_alert += "This rapid shift from oversold to overbought suggests a powerful surge in market participation."

# 4. Build the final Telegram message
msg = f"<b>📊 MCCLELLAN RECENT TREND</b>\n"
msg += f"Date: {recent_date}\n\n"

msg += "<b>Last 5 Sessions:</b>\n"
for date, val in mcclellan_recent.tail(5).items():
    # Use a special bullet if this was the day of the thrust
    prefix = "⚡️ " if (val == current_val and thrust_detected) else "• "
    msg += f"{prefix}{date.strftime('%m-%d')}: {val:+.1f}\n"

msg += thrust_alert

msg += f"\n\n<b>Current Status:</b>\n"
msg += f"• Oscillator: {current_val:+.1f}\n"
msg += f"• Prior Day: {prior_val:+.1f}\n"
msg += f"• 14D Change: {current_val - mcclellan_recent.iloc[0]:+.1f}"


In [11]:
# Use the historical series for the chart (from your combined 'df')
historical_mcclellan = fetch_mcclellan_oscillator(days=90)

In [12]:
# Send everything
send_telegram_with_chart(msg, historical_mcclellan)

✓ Chart and Message sent!


In [13]:
# Cell: VIX Data Fetching
import pandas as pd
import numpy as np
from yahooquery import Ticker
from datetime import datetime, timedelta

def get_vix_data():
    vix = Ticker('^VIX')
    vix_summary = vix.summary_detail.get('^VIX', {})
    
    vix_current = vix_summary.get('regularMarketPreviousClose', np.nan)
    vix_open = vix_summary.get('regularMarketOpen', np.nan)
    vix_day_high = vix_summary.get('regularMarketDayHigh', np.nan)
    vix_day_low = vix_summary.get('regularMarketDayLow', np.nan)
    
    history = vix.history(period="1wk")
    
    if isinstance(history, pd.DataFrame) and not history.empty:
        history = history.reset_index()
        history['date'] = pd.to_datetime(history['date'])
        
        week_high = history['high'].max()
        week_low = history['low'].min()
        week_open = history['close'].iloc[0] if len(history) > 0 else np.nan
        week_change_pct = ((vix_current - week_open) / week_open * 100) if week_open and week_open != 0 and not np.isnan(week_open) else np.nan
        
        prior_close = history['close'].iloc[-2] if len(history) > 1 else np.nan
    else:
        week_high = week_low = week_open = week_change_pct = prior_close = np.nan
    
    return {
        'current': vix_current,
        'prior': prior_close,
        'day_high': vix_day_high,
        'day_low': vix_day_low,
        'week_high': week_high,
        'week_low': week_low,
        'week_change_pct': week_change_pct,
        'history': history
    }

vix_data = get_vix_data()
print(f"Current VIX: {vix_data['current']}")
print(f"Week High: {vix_data['week_high']}, Week Low: {vix_data['week_low']}")
print(f"Week Change: {vix_data['week_change_pct']:.2f}%")

Current VIX: 22.22
Week High: 23.34000015258789, Week Low: 15.180000305175781
Week Change: 44.29%


In [14]:
# Cell: VIX Analysis and Telegram Alert
VIX_THRESHOLD = 30
VIX_NEAR_THRESHOLD = 28

def analyze_vix(vix_data):
    current = vix_data['current']
    prior = vix_data['prior']
    week_change = vix_data['week_change_pct']
    
    if current >= VIX_THRESHOLD:
        status = "🔴 HIGH FEAR"
        alert_type = "extreme"
    elif current >= VIX_NEAR_THRESHOLD:
        status = "🟡 NEAR THRESHOLD"
        alert_type = "warning"
    else:
        status = "🟢 NORMAL"
        alert_type = None
    
    trend = "📈 RISING" if current > prior else "📉 FALLING" if current < prior else "➡️ FLAT"
    
    message = f"<b>📊 VIX ANALYSIS (1 Week)</b>\n\n"
    message += f"<b>Current:</b> {current:.2f}\n"
    message += f"<b>Prior:</b> {prior:.2f}\n"
    message += f"<b>Day Range:</b> {vix_data['day_low']:.2f} - {vix_data['day_high']:.2f}\n"
    message += f"<b>Week Range:</b> {vix_data['week_low']:.2f} - {vix_data['week_high']:.2f}\n"
    message += f"<b>Week Change:</b> {week_change:+.2f}%\n\n"
    message += f"<b>Status:</b> {status}\n"
    message += f"<b>Trend:</b> {trend}\n"
    
    if alert_type == "extreme":
        message += f"\n⚠️ <b>ALERT:</b> VIX has exceeded {VIX_THRESHOLD}! High market fear/volatility expected."
    elif alert_type == "warning":
        message += f"\n⚠️ <b>WARNING:</b> VIX is nearing {VIX_THRESHOLD}. Monitor closely."
    
    return message, alert_type

vix_message, alert_type = analyze_vix(vix_data)
print(vix_message)

<b>📊 VIX ANALYSIS (1 Week)</b>

<b>Current:</b> 22.22
<b>Prior:</b> 22.22
<b>Day Range:</b> 20.53 - 21.25
<b>Week Range:</b> 15.18 - 23.34
<b>Week Change:</b> +44.29%

<b>Status:</b> 🟢 NORMAL
<b>Trend:</b> 📈 RISING



In [15]:
# Cell: VIX Chart and Telegram Send
import matplotlib.pyplot as plt
import requests
import os

def send_vix_telegram_chart(message, history_df):
    TELEGRAM_CHAT_ID = "972451274"
    if not bot_token or not TELEGRAM_CHAT_ID:
        print("⚠ Telegram credentials missing.")
        return

    chart_path = "vix_chart.png"
    try:
        plt.figure(figsize=(10, 5))
        plt.style.use('dark_background')
        
        if isinstance(history_df, pd.DataFrame) and not history_df.empty:
            history_df['date'] = pd.to_datetime(history_df['date'])
            plt.plot(history_df['date'], history_df['close'], color='#00d1ff', linewidth=2, label='VIX')
            plt.axhline(30, color='red', linestyle='--', linewidth=2, alpha=0.8, label=f'Threshold ({VIX_THRESHOLD})')
            plt.axhline(28, color='orange', linestyle=':', linewidth=1.5, alpha=0.6, label=f'Near Threshold ({VIX_NEAR_THRESHOLD})')
            plt.fill_between(history_df['date'], history_df['close'], 30, 
                            where=history_df['close'] >= 30, color='red', alpha=0.3)
        else:
            plt.text(0.5, 0.5, 'No historical data available', ha='center', va='center')
        
        plt.title(f"VIX - 1 Week Trend (Current: {vix_data['current']:.2f})")
        plt.xlabel('Date')
        plt.ylabel('VIX')
        plt.legend(loc='upper right')
        plt.grid(alpha=0.1)
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        plt.savefig(chart_path, bbox_inches='tight')
        plt.close()

        url = f"https://api.telegram.org/bot{bot_token}/sendPhoto"
        with open(chart_path, 'rb') as photo:
            payload = {
                'chat_id': TELEGRAM_CHAT_ID,
                'caption': message,
                'parse_mode': 'HTML'
            }
            files = {'photo': photo}
            try:
                r = requests.post(url, data=payload, files=files)
                if r.status_code == 200:
                    print("✓ VIX Chart sent to Telegram!")
                else:
                    print(f"✗ Failed: {r.text}")
            except Exception as e:
                print(f"✗ Connection Error: {e}")
    finally:
        if os.path.exists(chart_path):
            os.remove(chart_path)

send_vix_telegram_chart(vix_message, vix_data['history'])

✓ VIX Chart sent to Telegram!


In [16]:
# Cell: Crash Protection Strategy — Daily Market Stress Check
import pandas as pd
import numpy as np
from yahooquery import Ticker


def compute_crash_signal(vix_data):
    """Compute crash_protection signals from VIX + VIX3M.

    Two-signal system:
      1. VIX level > 25 (3-day smoothed)
      2. VIX3M/VIX ratio < 0.9 (term structure approaching backwardation)

    When BOTH fire → STRESS (reduce to 30% position).
    When VIX > 40 → CRASH (exit all positions).
    """
    # Fetch VIX3M data
    vix3m_ticker = Ticker('^VIX3M')
    vix3m_summary = vix3m_ticker.summary_detail.get('^VIX3M', {})
    vix3m_current = vix3m_summary.get('regularMarketPreviousClose', np.nan)
    vix3m_history = vix3m_ticker.history(period="5d")

    # Get current VIX
    vix_current = vix_data.get('current', np.nan)

    # Compute 3-day smoothed VIX from history if available
    vix_smooth = vix_current
    if isinstance(vix_data.get('history'), pd.DataFrame) and not vix_data['history'].empty:
        closes = vix_data['history']['close']
        if len(closes) >= 3:
            vix_smooth = closes.tail(3).mean()
        else:
            vix_smooth = closes.mean()

    vix_smooth = float(vix_smooth)

    # Signal 1: VIX level stress (VIX > 25)
    level_stress = vix_smooth > 25

    # Signal 2: Term structure stress (VIX3M/VIX < 0.9 = backwardation)
    ratio = float(vix3m_current / vix_smooth) if vix_smooth > 0 else 99.0
    term_stress = ratio < 0.9

    # Combined stress: both signals must fire
    stress = level_stress and term_stress
    vix_extreme = vix_smooth > 40

    signals = {
        'vix_smoothed': round(vix_smooth, 2),
        'vix3m': round(float(vix3m_current), 2),
        'vix3m_vix_ratio': round(ratio, 3),
        'level_stress': level_stress,
        'term_stress': term_stress,
        'combined_stress': stress,
        'vix_extreme': vix_extreme,
    }

    if vix_extreme:
        return 'CRASH', signals
    elif stress:
        return 'STRESS', signals
    elif vix_smooth > 25:
        return 'WARNING', signals
    else:
        return 'NORMAL', signals


def build_crash_message(status, signals):
    """Build formatted Telegram message for crash protection alert."""
    status_labels = {
        'CRASH': '🔴🚨 CRASH MODE — EXIT ALL POSITIONS',
        'STRESS': '🟠 MARKET STRESS — REDUCE TO 30%',
        'WARNING': '🟡 VIX ELEVATED - Monitor',
        'NORMAL': '🟢 NORMAL - Stay invested',
    }

    msg = f"🛡️ CRASH PROTECTION\n\n"
    msg += f"Status: {status_labels.get(status, status)}\n\n"
    msg += "Key Metrics:\n"
    msg += f"• VIX (3d avg): {signals['vix_smoothed']}\n"
    msg += f"• VIX3M: {signals['vix3m']}\n"
    msg += f"• VIX3M/VIX: {signals['vix3m_vix_ratio']}\n\n"
    msg += "Signals:\n"
    msg += f"• VIX > 25: {'✅ YES' if signals['level_stress'] else '❌ no'}\n"
    msg += f"• Ratio < 0.9: {'✅ YES' if signals['term_stress'] else '❌ no'}\n"
    msg += f"• Combined: {'✅ ACTIVE' if signals['combined_stress'] else '❌ none'}\n\n"

    if status == 'CRASH':
        msg += "🚨 ACTION: EXIT ALL SPY POSITIONS. VIX > 40."
    elif status == 'STRESS':
        msg += "⚠️ ACTION: Reduce SPY exposure to 30%. Both VIX level and term structure signal stress."
    elif status == 'WARNING':
        msg += "📋 Monitor: VIX elevated but no backwardation. No action needed yet."
    else:
        msg += "✅ All clear. Stay fully invested."

    return msg


# ---- Execute --------------------------------------------------------
status, signals = compute_crash_signal(vix_data)
crash_msg = build_crash_message(status, signals)
print(crash_msg)
send_telegram_message(crash_msg)

🛡️ CRASH PROTECTION

Status: 🟢 NORMAL - Stay invested

Key Metrics:
• VIX (3d avg): 21.0
• VIX3M: 21.31
• VIX3M/VIX: 1.015

Signals:
• VIX > 25: ❌ no
• Ratio < 0.9: ❌ no
• Combined: ❌ none

✅ All clear. Stay fully invested.
Message sent successfully!
